# Stratosphere Impact — Skill Assessment Notebook

Seasonal forecast evaluation against ERA5 — NDJFMA focus.  
Variables: `tas · uas · tasmax · tasmin · z500 · psl`.
Metrics   : ACC · RMSE · MSSS · BIAS · BSS (terciles) · NAO (Li & Wang 2003).
Data      : monthly, `[time, lat, lon]` ERA5 / `[member, time, lat, lon]` experiments

### · Needed Configuration 


In [ ]:

from pyexpat import EXPAT_VERSION

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
import netCDF4 as nc
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy import stats as spstats
import gc
import os
import matplotlib as mpl
from skill_tools import *

mpl.rcParams.update({
    'figure.dpi': 120,
    'font.size': 12,
    'axes.titlesize': 12,
    'axes.labelsize': 12,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
})

DATA_DIR = {
    "ERA5"   : "/path_to ERA5/",
    "Free"   : "/path_to_exp/",
    "STClim" : "/path_to_exp/",
    "STTrop" : "/path_to_exp/",
    "STPerfect" : "/path_to_exp/"
    
}
EXPERIMENTS = ["Free", "STClim","STTrop","STPerfect"]
EXP1 = {
    "Free"   : "rfph",
    "STClim" : "rfphcl",
    "STTrop" : "rfphst",
    "STPerfect" : "rfphst"

}
EXP1_suffix = {
    "Free"   : "",
    "STClim" : "",
    "STTrop" : "_tropics_qboi",
    "STPerfect" : "_global"

}

VARIABLES   = ["u10","tas", "uas", "pr", "z500", "psl"]
EXP_COLORS  = {
    "Free"   : "#2196F3",
    "STClim" : "#F44336",
    "STTrop" : "#4CAF50",
    "STPerfect" : "#AF824C",
}

DATA_MODE = {
    "Free"   : "emean",
    "STClim" : "emean",
    "STTrop" : "emean",
    "STPerfect" : "emean"
    
}
SIG_LEVEL = 0.05

NC_VARNAME = {
    "u10" : {"ERA5": "u", "model": "ua"},
    "tas" : {"ERA5": "tas", "model": "tas"},
    "uas" : {"ERA5": "uas", "model": "uas"},
    "pr"  : {"ERA5": "pr",  "model": "pr"},
    "z500": {"ERA5": "z",   "model": "zg"},   # ← aquí
    "psl" : {"ERA5": "psl", "model": "psl"},
}

ERA5_START_YEAR  = 1980
ERA5_END_YEAR    = 2021
MODEL_START_YEAR = 1980
MODEL_END_YEAR   = 2020
COMMON_YEARS     = np.arange(MODEL_START_YEAR, MODEL_END_YEAR )

NCOMMON          = len(COMMON_YEARS)

LEAD_MONTHS = np.arange(6)
LEAD_LABELS = ["Nov", "Dec", "Jan", "Feb", "Mar", "Apr"]
N_LEADS     = len(LEAD_MONTHS)

NAO_SOUTH = {"lat": 36.0, "lon": -28.0}
NAO_NORTH = {"lat": 65.0, "lon": -20.0}

# ── Regional Domain for plotting ────────────────────────────────────────────────────────
DOMAINS = {
    "global"      : [0.0,  357.5, -90.0,  90.0],   # lon0, lon1, lat0, lat1
    "euro_atlantic": [-80.0, 60.0,  10.0,  90.0],
}

results     = {exp: {var: {} for var in VARIABLES} for exp in EXPERIMENTS}
pvals       = {exp: {var: {} for var in VARIABLES} for exp in EXPERIMENTS}
nao_results = {exp: {} for exp in EXPERIMENTS}

OUTDIR = "/skill_figures"
os.makedirs(OUTDIR, exist_ok=True)

print("Configuration loaded ✓")
print(f"  Experiments  : {EXPERIMENTS}")
print(f"  Variables    : {VARIABLES}")
print(f"  Common years : {COMMON_YEARS[0]}–{COMMON_YEARS[-1]}  ({NCOMMON} years)")
print(f"  Leads        : {dict(zip(LEAD_MONTHS, LEAD_LABELS))}")


### · Climate indices — user-provided year arrays


In [ ]:

ENSO_NINO  =  np.array([1982, 1986, 1991, 1994, 1997, 2009, 2015])   #  (Nov)
ENSO_NINA  = np.array([1984, 1988, 1998, 1999, 2007, 2010, 2020])
QBO_EQBO   = np.array([1981,1984, 1989, 1994, 1996, 1998, 2001, 2003, 2005, 2007, 2012, 2014,2018])
QBO_WQBO = np.array([1980, 1982, 1985, 1987, 1988, 1990, 1995, 1997, 1999, 2002, 2004, 2006, 2008, 2010, 2013, 2015,2016,2020])


def year_to_enso(yr):
    if yr in ENSO_NINO: return "NINO"
    if yr in ENSO_NINA: return "NINA"
    return "NEUT"

def year_to_qbo(yr):
    if yr in QBO_WQBO: return "WQBO"
    if yr in QBO_EQBO: return "EQBO"
    return "NEUT"

ENSO_COLOR = {"NINO": "#E74C3C", "NINA": "#2980B9", "NEUT": "0.85"}
QBO_COLOR  = {"WQBO": "#8E44AD", "EQBO": "#F39C12", "NEUT": "0.85"}

print("Climate indices loaded")
print(f"  El Niño years : {ENSO_NINO}")
print(f"  La Niña years : {ENSO_NINA}")
print(f"  WQBO years    : {QBO_WQBO}")
print(f"  EQBO years    : {QBO_EQBO}")

###  · Load ERA5


In [ ]:
era5 = {}   # era5[var] = {"data": ndarray, "lat": ndarray, "lon": ndarray}

for var in VARIABLES:
    ncvar = NC_VARNAME[var]["ERA5"]
    fpath = os.path.join(DATA_DIR["ERA5"], f"{var}_mon_ERA5_1980-2020.nc")
    with nc.Dataset(fpath) as ds:
        data  = ds.variables[ncvar][:].data.astype(np.float32)
        data = np.squeeze(data)
        era5[var] = {
            "data": data,
            "lat" : ds.variables["lat"][:].data.astype(np.float64),
            "lon" : ds.variables["lon"][:].data.astype(np.float64),
        }
    sh = era5[var]["data"].shape
    print(f"  ERA5 {var:8s}: shape={sh}  "
          f"lat=[{era5[var]['lat'][0]:.1f},{era5[var]['lat'][-1]:.1f}]  "
          f"lon=[{era5[var]['lon'][0]:.1f},{era5[var]['lon'][-1]:.1f}]")

# Shared ERA5 grid (assumed identical across variables)
ERA5_LAT = era5[VARIABLES[0]]["lat"]
ERA5_LON = era5[VARIABLES[0]]["lon"]

gc.collect()
print(f"\nERA5 grid: {len(ERA5_LAT)}×{len(ERA5_LON)} ")


###  · Load Experiments


In [ ]:

exps = {}   # exps[exp][var] = {"data": ndarray, "lat": ndarray, "lon": ndarray}

for exp in EXPERIMENTS:
    exps[exp] = {}
    mode = DATA_MODE[exp]
    exp1 = EXP1[exp]
    exp2 = EXP1_suffix[exp]
    
 
    for var in VARIABLES:
        ncvar = NC_VARNAME[var]["model"]

        if mode == "members":
            fpath = os.path.join(DATA_DIR[exp], f"mean_{var}_Amon_{exp.lower()}{exp1}_1980-2020.nc")
            with nc.Dataset(fpath) as ds:
                data = ds.variables[ncvar][:].data.astype(np.float32)
                data = np.squeeze(data)   # in case of single-member experiments
                lat  = ds.variables["lat"][:].data.astype(np.float64)
                lon  = ds.variables["lon"][:].data.astype(np.float64)
            exps[exp][var] = {
                "mode"  : "members",
                "data"  : data,
                "emean" : np.nanmean(data, axis=0),  # [ntime, nlat, nlon]
                "spread": np.nanstd(data, axis=0),
                "lat"   : lat, "lon": lon,
            }
            sh = data.shape
            print(f"  {exp} {var:8s} [members]: {sh}  ({sh[0]} members)")
 
        elif mode == "emean":
            f_mean  = os.path.join(DATA_DIR[exp], f"mean_{var}_Amon_{exp1.lower()}_1980-2020{exp2}.nc")
            f_std   = os.path.join(DATA_DIR[exp], f"std_{var}_Amon_{exp1.lower()}_1980-2020{exp2}.nc")
            with nc.Dataset(f_mean) as ds:
                emean = ds.variables[ncvar][:].data.astype(np.float32)
                emean = np.squeeze(emean)
                lat   = ds.variables["lat"][:].data.astype(np.float64)
                lon   = ds.variables["lon"][:].data.astype(np.float64)
            with nc.Dataset(f_std) as ds:
                spread = ds.variables[ncvar][:].data.astype(np.float32)
                spread = np.squeeze(spread)

            exps[exp][var] = {
                "mode"  : "emean",
                "data"  : None,
                "emean" : emean,
                "spread": spread,
                "lat"   : lat, "lon": lon,
            }
            print(f"  {exp} {var:8s} [emean ]: mean{emean.shape}  spread{spread.shape}")
 
        else:
            raise ValueError(f"Unknown DATA_MODE '{mode}' for {exp}")
 
    print()
 
gc.collect()


### · Regrid Models to ERA5 Grid


In [ ]:
for exp in EXPERIMENTS:
    src_lat = exps[exp][VARIABLES[0]]["lat"]
    src_lon = exps[exp][VARIABLES[0]]["lon"]

    same_grid = (
        src_lat.shape == ERA5_LAT.shape and
        src_lon.shape == ERA5_LON.shape and
        np.allclose(src_lat, ERA5_LAT, atol=1e-4) and
        np.allclose(src_lon, ERA5_LON, atol=1e-4)
    )

    if same_grid:
        print(f"{exp}: already on ERA5 grid — skipping.")
        continue

    print(f"Regridding {exp}  "
          f"({len(src_lat)}×{len(src_lon)}) → ({len(ERA5_LAT)}×{len(ERA5_LON)}) ...")

    for var in VARIABLES:
        mode = exps[exp][var]["mode"]
 
        if mode == "members":
            d    = exps[exp][var]["data"]                    # [mem, time, nlat, nlon]
            d_rg = bilinear_interp(d, src_lat, src_lon, ERA5_LAT, ERA5_LON)
            exps[exp][var]["data"]   = d_rg
            exps[exp][var]["emean"]  = np.nanmean(d_rg, axis=0)
            exps[exp][var]["spread"] = np.nanstd(d_rg, axis=0)
            print(f"  {var}: {d.shape} → {d_rg.shape}")
            del d, d_rg
 
        else:   # emean — regrid mean_ and std_ independently
            em = exps[exp][var]["emean"]                     # [time, nlat, nlon]
            sp = exps[exp][var]["spread"]
            em_rg = bilinear_interp(em, src_lat, src_lon, ERA5_LAT, ERA5_LON)
            sp_rg = bilinear_interp(sp, src_lat, src_lon, ERA5_LAT, ERA5_LON)
            exps[exp][var]["emean"]  = em_rg
            exps[exp][var]["spread"] = sp_rg
            print(f"  {var} (emean ): {em.shape} → {em_rg.shape}")
            del em, sp, em_rg, sp_rg
 
        exps[exp][var]["lat"] = ERA5_LAT
        exps[exp][var]["lon"] = ERA5_LON
        gc.collect()
 
    print(f"  {exp} done \n")
 
gc.collect()
 

###  · Extract NDJFMA at Each Lead Time


In [ ]:
# Build aligned `[nyr, …]` arrays for the common 1980–2020 period.
#
# | Lead | Month | Calendar       |
# |------|-------|----------------|
# | 0    | Nov   | year Y         |
# | 1    | Dec   | year Y         |
# | 2    | Jan   | year Y+1       |
# | 3    | Feb   | year Y+1       |
# | 4    | Mar   | year Y+1       |
# | 5    | Apr   | year Y+1       |
# obs_lead[var][lead]        → [nyr, nlat, nlon]
# fcst_lead[exp][var][lead]  → [nyr, nmem, nlat, nlon]

obs_lead  = {var: {} for var in VARIABLES}
fcst_lead = {exp: {var: {} for var in VARIABLES} for exp in EXPERIMENTS}
emean_lead  = {exp: {var: {} for var in VARIABLES} for exp in EXPERIMENTS}
spread_lead = {exp: {var: {} for var in VARIABLES} for exp in EXPERIMENTS}


for var in VARIABLES:
    e5 = era5[var]["data"]    # [ntime_era5, nlat, nlon]

    for lead in LEAD_MONTHS:
        e5_idx = era5_flat_index(COMMON_YEARS, ERA5_START_YEAR, lead)
        obs_lead[var][lead] = e5[e5_idx]                      # [nyr, nlat, nlon]

    for exp in EXPERIMENTS:
        mode = DATA_MODE[exp]
        for lead in LEAD_MONTHS:
            m_idx = model_flat_index(COMMON_YEARS, MODEL_START_YEAR, lead)
 
            if mode == "members":
                md  = exps[exp][var]["data"]                  # [nmem, ntime, nlat, nlon]
                tmp = md[:, m_idx, :, :]                      # [nmem, nyr, nlat, nlon]
                fcst = tmp.transpose(1, 0, 2, 3)              # [nyr, nmem, nlat, nlon]
                fcst_lead[exp][var][lead]   = fcst
                emean_lead[exp][var][lead]  = np.nanmean(fcst, axis=1)
                spread_lead[exp][var][lead] = np.nanstd(fcst, axis=1)
 
            else:   # emean
                em = exps[exp][var]["emean"]                  # [ntime, nlat, nlon]
                sp = exps[exp][var]["spread"]
                fcst_lead[exp][var][lead]   = None            # no raw members
                emean_lead[exp][var][lead]  = em[m_idx]       # [nyr, nlat, nlon]
                spread_lead[exp][var][lead] = sp[m_idx]
 
    print(f"  {var}: extracted {NCOMMON} yrs × {N_LEADS} leads ")
 
gc.collect()
print("\nLead-time extraction complete ")


### · Climatology & Anomalies


In [ ]:
clim_obs  = {var: {} for var in VARIABLES}
anom_obs  = {var: {} for var in VARIABLES}
anom_fcst = {exp: {var: {} for var in VARIABLES} for exp in EXPERIMENTS}

for var in VARIABLES:
    for lead in LEAD_MONTHS:
        obs = obs_lead[var][lead]                        # [nyr, nlat, nlon]
        clim_obs[var][lead] = np.nanmean(obs, axis=0)
        anom_obs[var][lead] = obs - clim_obs[var][lead]

        for exp in EXPERIMENTS:
            em     = emean_lead[exp][var][lead]          # [nyr, nlat, nlon]
            clim_f = np.nanmean(em, axis=0)
            anom_fcst[exp][var][lead] = em - clim_f
        gc.collect()

    print(f"  {var}: climatology & anomalies done ")

emean_fcst = anom_fcst   
print("\nAnomalies ready ")


### · Compute All Skill Metrics and Select Case


In [ ]:
###SELECT CASE
CASE_YEARS = ENSO_NINO-1993 
for exp in EXPERIMENTS:
    mode = DATA_MODE[exp]
    for var in VARIABLES:
        for metric in ["acc", "rmse", "msss", "bias", "bss_below", "bss_above"]:
            results[exp][var][metric] = {}
        for metric in ["acc", "bias", "bss_below", "bss_above"]:
            pvals[exp][var][metric] = {}
 
        for lead in LEAD_MONTHS:
            oa      = anom_obs[var][lead][CASE_YEARS,:,:]                # [nyr, nlat, nlon]
            fa      = anom_fcst[exp][var][lead][CASE_YEARS,:,:]               # [nyr, nlat, nlon]
            obs_abs = obs_lead[var][lead][CASE_YEARS,:,:]                     # [nyr, nlat, nlon]
            em_abs  = emean_lead[exp][var][lead][CASE_YEARS,:,:]              # [nyr, nlat, nlon]
 
            # ── Deterministic metrics ─────────────────────────────────────────
            r_map = acc(fa, oa)
            results[exp][var]["acc"][lead]  = r_map
            results[exp][var]["rmse"][lead] = rmse(fa, oa)
            results[exp][var]["msss"][lead] = msss(fa, oa)
            results[exp][var]["bias"][lead] = bias(em_abs, obs_abs)
 
            pvals[exp][var]["acc"][lead]  = acc_pvalue(r_map, NCOMMON)
            pvals[exp][var]["bias"][lead] = bias_pvalue(em_abs - obs_abs)
 
            # ── Probabilistic metrics (BSS) ───────────────────────────────────
            sp = spread_lead[exp][var][lead][CASE_YEARS,:,:]   # [nyr, nlat, nlon]
            if mode == "members":
                bss = bss_tercile(fcst_lead[exp][var][lead], obs_abs)
            else:
                bss = bss_tercile(None, obs_abs,
                                  fcst_emean=em_abs, fcst_spread=sp)
 
            results[exp][var]["bss_below"][lead] = bss["bss_below"]
            results[exp][var]["bss_above"][lead] = bss["bss_above"]
            pvals[exp][var]["bss_below"][lead]   = bss["p_below"]
            pvals[exp][var]["bss_above"][lead]   = bss["p_above"]
 
        print(f"  {exp} {var}: metrics + significance done [{mode}]")
    gc.collect()
 
print("\nAll skill metrics computed")
 


In [ ]:
## check keys
print(pvals[EXPERIMENTS[0]][VARIABLES[0]].keys())
# Should return: dict_keys(['acc', 'bias', 'bss_below', 'bss_above'])

## Skill Maps — ACC (stereoplots)


In [ ]:
# ── ACC maps —  ─────────────────────────────────────────────────────────────
## Select variable to plot
var = "tas"
cmap_acc = plt.cm.RdYlGn
vmin_acc, vmax_acc = -1, 1
 
fig, axes = make_map_axes(len(EXPERIMENTS), N_LEADS,proj=PROJ_NH,height_per_row=2.5,title=f"ACC — {var.upper()} | NDJFMA (dots = not significant p<{SIG_LEVEL}) | NH")
 
for r, exp in enumerate(EXPERIMENTS):
    for c, lead in enumerate(LEAD_MONTHS):
        ax  = axes[r, c]
        sig = sig_mask(pvals[exp][var]["acc"][lead])
        im  = skill_map(ax, results[exp][var]["acc"][lead],
                        ERA5_LAT, ERA5_LON, cmap_acc, vmin_acc, vmax_acc,
                        title=f"{exp} | {LEAD_LABELS[c]}", sig=sig,proj=PROJ_NH)
        if c == 0:
            ax.set_ylabel(exp, fontsize=9)
 
add_colorbar(fig, im, axes, label="ACC", orientation="horizontal")
plt.savefig(f"{OUTDIR}/acc_{var}.png", dpi=150, bbox_inches="tight")
plt.show(); gc.collect()
 


###  ACC values for different variables and months 


In [ ]:
# ── Región NAE ────────────────────────────────────────────────────────────────
NAE_LAT0, NAE_LAT1 =  30,  80
NAE_LON0, NAE_LON1 = -60,  40

lon_norm    = np.where(ERA5_LON > 180, ERA5_LON - 360, ERA5_LON)
lat_mask_nae = (ERA5_LAT  >= NAE_LAT0) & (ERA5_LAT  <= NAE_LAT1)
lon_mask_nae = (lon_norm   >= NAE_LON0) & (lon_norm   <= NAE_LON1)
lat_nae      = ERA5_LAT[lat_mask_nae]

def area_mean_nae(data2d, lat_nae, lat_mask, lon_mask):
    """ACC [nlat,nlon] → escalar media ponderada en NAE."""
    d = data2d[lat_mask, :][:, lon_mask]
    w = np.cos(np.deg2rad(lat_nae))
    w_2d = np.broadcast_to(w[:, np.newaxis], d.shape)
    return np.nansum(d * w_2d) / np.nansum(w_2d * np.isfinite(d))

# ── Tabla ACC NAE por experimento, variable y lead ────────────────────────────
print(f"\nACC — NAE region [{NAE_LAT0}-{NAE_LAT1}°N, {NAE_LON0}-{NAE_LON1}°E]")
print(f"{'':20s} " + " ".join(f"{l:>6}" for l in LEAD_LABELS))
print("─" * (20 + 7 * N_LEADS))

for exp in EXPERIMENTS:
    for var in VARIABLES:
        accs = []
        for l in LEAD_MONTHS:
            val = area_mean_nae(results[exp][var]["acc"][l],
                                lat_nae, lat_mask_nae, lon_mask_nae)
            accs.append(val)
        row = " ".join(f"{v:6.3f}" for v in accs)
        print(f"  {exp:10s} {var:6s}: {row}")
    print()

###  ACC maps over the NAE region 

In [ ]:
## Select variable to plot
var = "psl"
cmap_acc = "RdBu_r"
vmin_acc, vmax_acc = -1, 1

fig, axes = make_map_axes(len(EXPERIMENTS[0:2]), N_LEADS, figsize=(16, 2 * len(EXPERIMENTS[0:2])),
                          title=f"ACC — {var.upper()} | NDJFMA  (dots = not significant p<{SIG_LEVEL})")
 
for r, exp in enumerate(EXPERIMENTS[0:2]):
    for c, lead in enumerate(LEAD_MONTHS):
        ax  = axes[r, c]
        sig = sig_mask(pvals[exp][var]["acc"][lead])
        im  = skill_map_nae(ax, results[exp][var]["acc"][lead],
                        ERA5_LAT, ERA5_LON, cmap_acc, vmin_acc, vmax_acc,
                        title=f"{exp} | {LEAD_LABELS[c]}", sig=sig)
        if c == 0:
            ax.set_ylabel(exp, fontsize=9)
 
add_colorbar(fig, im, axes, label="ACC")
plt.savefig(f"{OUTDIR}/acc_{var}.png", dpi=150, bbox_inches="tight")
plt.show(); gc.collect()
 


In [ ]:
var = "psl"
 
fig, axes = make_map_axes(len(EXPERIMENTS), N_LEADS, figsize=(16, 2 * len(EXPERIMENTS)),
                          title=f"ACC — {var.upper()} | NDJFMA  (dots = not significant p<{SIG_LEVEL})")
 
for r, exp in enumerate(EXPERIMENTS):
    for c, lead in enumerate(LEAD_MONTHS):
        ax  = axes[r, c]
        sig = sig_mask(pvals[exp][var]["acc"][lead])
        im  = skill_map_nae(ax, results[exp][var]["acc"][lead],
                        ERA5_LAT, ERA5_LON, cmap_acc, vmin_acc, vmax_acc,
                        title=f"{exp} | {LEAD_LABELS[c]}", sig=sig)
        if c == 0:
            ax.set_ylabel(exp, fontsize=9)
 
add_colorbar(fig, im, axes, label="ACC")
plt.savefig(f"{OUTDIR}/acc_{var}.png", dpi=150, bbox_inches="tight")
plt.show(); gc.collect()
 


### ACC Global maps 

In [ ]:
for var in ["uas"]:
    fig, axes = make_map_axes(len(EXPERIMENTS), N_LEADS,
                              figsize=(16, 2 * len(EXPERIMENTS)),
                              title=f"ACC — {var.upper()} | NDJFMA  (dots = not significant p<{SIG_LEVEL})")
    for r, exp in enumerate(EXPERIMENTS):
        for c, lead in enumerate(LEAD_MONTHS):
            ax  = axes[r, c]
            sig = sig_mask(pvals[exp][var]["acc"][lead])
            im  = skill_map(ax, results[exp][var]["acc"][lead],
                            ERA5_LAT, ERA5_LON, cmap_acc, vmin_acc, vmax_acc,
                            title=f"{exp} | {LEAD_LABELS[c]}", sig=sig)
            if c == 0:
                ax.set_ylabel(exp, fontsize=9)
    add_colorbar(fig, im, axes, label="ACC")
    plt.savefig(f"{OUTDIR}/acc_{var}.png", dpi=150, bbox_inches="tight")
    plt.show()
    gc.collect()
 


### Skill Maps — RMSE


In [ ]:
cmap_rmse = plt.cm.YlOrRd
 
for var in VARIABLES:
    # Determine robust vmax from data
    all_rmse = np.stack([results[EXPERIMENTS[0]][var]["rmse"][l] for l in LEAD_MONTHS])
    vmax_rmse = float(np.nanpercentile(np.abs(all_rmse), 95))
 
    fig, axes = make_map_axes(len(EXPERIMENTS), N_LEADS,
                              figsize=(16, 2 * len(EXPERIMENTS)),
                              title=f"RMSE — {var.upper()} | NDJFMA")
    for r, exp in enumerate(EXPERIMENTS):
        for c, lead in enumerate(LEAD_MONTHS):
            ax = axes[r, c]
            im = skill_map(ax, results[exp][var]["rmse"][lead],
                           ERA5_LAT, ERA5_LON, cmap_rmse, 0, vmax_rmse,
                           title=f"{exp} | {LEAD_LABELS[c]}")
            if c == 0:
                ax.set_ylabel(exp, fontsize=9)
    add_colorbar(fig, im, axes, label="RMSE")
    plt.savefig(f"{OUTDIR}/rmse_{var}.png", dpi=150, bbox_inches="tight")
    plt.show()
    gc.collect()
 


### Skill Maps — MSSS

In [ ]:
cmap_msss = plt.cm.RdYlGn
vmin_msss, vmax_msss = -1, 1
 
for var in VARIABLES:
    fig, axes = make_map_axes(len(EXPERIMENTS), N_LEADS,
                              figsize=(16, 2 * len(EXPERIMENTS)),
                              title=f"MSSS — {var.upper()} | NDJFMA")
    for r, exp in enumerate(EXPERIMENTS):
        for c, lead in enumerate(LEAD_MONTHS):
            ax = axes[r, c]
            im = skill_map(ax, results[exp][var]["msss"][lead],
                           ERA5_LAT, ERA5_LON, cmap_msss, vmin_msss, vmax_msss,
                           title=f"{exp} | {LEAD_LABELS[c]}")
            if c == 0:
                ax.set_ylabel(exp)
    add_colorbar(fig, im, axes, label="MSSS")
    plt.savefig(f"{OUTDIR}/msss_{var}.png", dpi=150, bbox_inches="tight")
    plt.show()
    gc.collect()
 




 # Skill Maps — BIAS
 

In [ ]:

cmap_bias = plt.cm.RdBu_r
 
for var in VARIABLES:
    all_bias = np.stack([results[EXPERIMENTS[0]][var]["bias"][l] for l in LEAD_MONTHS])
    vlim = float(np.nanpercentile(np.abs(all_bias), 97))
 
    fig, axes = make_map_axes(len(EXPERIMENTS), N_LEADS,
                              figsize=(16, 2 * len(EXPERIMENTS)),
                              title=f"BIAS — {var.upper()} | NDJFMA  (dots = not significant p<{SIG_LEVEL})")
    for r, exp in enumerate(EXPERIMENTS):
        for c, lead in enumerate(LEAD_MONTHS):
            ax  = axes[r, c]
            sig = sig_mask(pvals[exp][var]["bias"][lead])
            im  = skill_map(ax, results[exp][var]["bias"][lead],
                            ERA5_LAT, ERA5_LON, cmap_bias, -vlim, vlim,
                            title=f"{exp} | {LEAD_LABELS[c]}", sig=sig)
            if c == 0:
                ax.set_ylabel(exp, fontsize=9)
    add_colorbar(fig, im, axes, label="Bias (model − ERA5)")
    plt.savefig(f"{OUTDIR}/bias_{var}.png", dpi=150, bbox_inches="tight")
    plt.show()
    gc.collect()
 
# %% [markdown]
# ---
# # Skill Maps — BSS (tercile categories)
 
# %%
cmap_bss = plt.cm.RdYlGn
vmin_bss, vmax_bss = -0.5, 0.5
 
for var in ["tas", "z500", "psl"]:   # extend as needed
    for cat, key in [("Below-normal", "bss_below"), ("Above-normal", "bss_above")]:
        fig, axes = make_map_axes(len(EXPERIMENTS), N_LEADS,
                                  figsize=(16, 2 * len(EXPERIMENTS)),
                                  title=f"BSS {cat} — {var.upper()} | NDJFMA  (dots = not significant p<{SIG_LEVEL})")
        for r, exp in enumerate(EXPERIMENTS):
            for c, lead in enumerate(LEAD_MONTHS):
                ax  = axes[r, c]
                sig = sig_mask(pvals[exp][var][key][lead])
                im  = skill_map(ax, results[exp][var][key][lead],
                                ERA5_LAT, ERA5_LON, cmap_bss, vmin_bss, vmax_bss,
                                title=f"{exp} | {LEAD_LABELS[c]}", sig=sig)
                if c == 0:
                    ax.set_ylabel(exp, fontsize=9)
        add_colorbar(fig, im, axes, label="BSS")
        plt.savefig(f"{OUTDIR}/bss_{key}_{var}.png", dpi=150, bbox_inches="tight")
        plt.show()
        gc.collect()
 


### Skill vs Lead Time — Deterministic Metrics
Area-weighted spatial mean of ACC, RMSE, MSSS per experiment.
 

In [ ]:
MASKS = {
    "global" : None,
    "NAE"    : make_region_mask(ERA5_LAT, ERA5_LON, 30,  80, -60,  40),
    "N.Atl"     : make_region_mask(ERA5_LAT, ERA5_LON, 30,  70, -80, -10),
    "EU"     : make_region_mask(ERA5_LAT, ERA5_LON, 35,  75, -10,  30),
    "Med"    : make_region_mask(ERA5_LAT, ERA5_LON, 25,  50, -10,  40),
}



# ── Compute area-weighted means and store for scorecard ──────────────────────
acc_lead  = {exp: {var: np.full(N_LEADS, np.nan) for var in VARIABLES} for exp in EXPERIMENTS}
rmse_lead = {exp: {var: np.full(N_LEADS, np.nan) for var in VARIABLES} for exp in EXPERIMENTS}
msss_lead = {exp: {var: np.full(N_LEADS, np.nan) for var in VARIABLES} for exp in EXPERIMENTS}

REGION = "NAE"

for exp in EXPERIMENTS:
    for var in VARIABLES:
        for lead in LEAD_MONTHS:
            acc_lead[exp][var][lead]  = float(area_mean(results[exp][var]["acc"][lead],  ERA5_LAT, MASKS[REGION]))
            rmse_lead[exp][var][lead] = float(area_mean(results[exp][var]["rmse"][lead], ERA5_LAT, MASKS[REGION]))
            msss_lead[exp][var][lead] = float(area_mean(results[exp][var]["msss"][lead], ERA5_LAT, MASKS[REGION]))

 
print("Area-weighted skill scores computed ")
 
# %%
# ── ACC vs lead — all variables, multi-experiment comparison ─────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
fig.suptitle("Area-weighted ACC vs Lead Time", fontsize=13, fontweight="bold")
 
for ax, var in zip(axes.flat, VARIABLES):
    for exp in EXPERIMENTS:
        ax.plot(LEAD_LABELS, acc_lead[exp][var], marker="o", lw=2,
                color=EXP_COLORS[exp], label=exp)
    ax.axhline(0, color="k", lw=0.8, ls="--", alpha=0.5)
    ax.axhline(0.5, color="gray", lw=0.6, ls=":", alpha=0.7)
    ax.set_title(var.upper(), fontweight="bold")
    ax.set_ylim(-0.3, 1.0)
    ax.set_ylabel("ACC")
    ax.set_xlabel("Lead month")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
 
plt.savefig(f"{OUTDIR}/acc_vs_lead_all_vars.png", dpi=150, bbox_inches="tight")
plt.show()
 
# %%
# ── RMSE vs lead ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
fig.suptitle("Area-weighted RMSE vs Lead Time", fontsize=13, fontweight="bold")
 
for ax, var in zip(axes.flat, VARIABLES):
    for exp in EXPERIMENTS:
        ax.plot(LEAD_LABELS, rmse_lead[exp][var], marker="o", lw=2,
                color=EXP_COLORS[exp], label=exp)
    ax.set_title(var.upper(), fontweight="bold")
    ax.set_ylabel("RMSE")
    ax.set_xlabel("Lead month")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
 
plt.savefig(f"{OUTDIR}/rmse_vs_lead_all_vars.png", dpi=150, bbox_inches="tight")
plt.show()
 
# %%
# ── MSSS vs lead ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
fig.suptitle("Area-weighted MSSS vs Lead Time", fontsize=13, fontweight="bold")
 
for ax, var in zip(axes.flat, VARIABLES):
    for exp in EXPERIMENTS:
        ax.plot(LEAD_LABELS, msss_lead[exp][var], marker="o", lw=2,
                color=EXP_COLORS[exp], label=exp)
    ax.axhline(0, color="k", lw=0.8, ls="--", alpha=0.5)
    ax.set_title(var.upper(), fontweight="bold")
    ax.set_ylim(-0.5, 1.0)
    ax.set_ylabel("MSSS")
    ax.set_xlabel("Lead month")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
 
plt.savefig(f"{OUTDIR}/msss_vs_lead_all_vars.png", dpi=150, bbox_inches="tight")
plt.show()
